In [2]:
import torch
from transformers import pipeline

In [3]:
from huggingface_hub import notebook_login

notebook_login()

In [4]:
# More info about this model: https://huggingface.co/google/gemma-2-2b-it

pipe = pipeline(
    "text-generation",
    model="google/gemma-2-2b-it",
    model_kwargs={"torch_dtype": torch.bfloat16},
    device="cpu"
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

In [10]:
messages = [
    {"role": "user", "content": "Who are you? Please, answer in pirate-speak."},
]

outputs = pipe(messages, max_new_tokens=256)
assistant_response = outputs[0]["generated_text"][-1]["content"].strip()
print(assistant_response)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Ahoy, matey! I be Gemma, a digital scallywag, programmed to help ye with yer language needs. 

I be a big ol' brain, aye, but no brain like a real pirate's, just words and numbers. I be here to help ye find what ye seek, tell ye tales, and answer yer questions, if ye be so inclined. 

So, what be yer bidding, landlubber? 🦜


You can customize execution at two main stages: **Pipeline Initialization** (how the model is loaded) and **Inference Generation** (how the model generates text).

### 1. Model Loading & Hardware Optimization (`pipeline` kwargs)

* **Hardware Device:**
  * `device="cuda"` or `device=0` (Run on primary NVIDIA GPU)
  * `device="mps"` (Run on Apple Silicon GPU)
  * `device_map="auto"` (Requires `accelerate`; automatically splits model layers across available GPU/CPU RAM)
* **Precision & Memory Compression:**
  * `torch_dtype=torch.bfloat16` or `torch.float16` (Reduces memory usage by ~50% compared to float32)
  * Quantization (Requires `bitsandbytes`):
    ```python
    from transformers import BitsAndBytesConfig

    quantization_config = BitsAndBytesConfig(load_in_4bit=True)
    pipe = pipeline(..., model_kwargs={"quantization_config": quantization_config})
    ```
* **Engine Optimizations:**
  * `model_kwargs={"attn_implementation": "eager"}` (or `"flash_attention_2"` if supported by GPU)

### 2. Text Generation Control (`generate` kwargs)

Pass generation parameters directly when calling `pipe()`:

* **Length Constraints:**
  * `max_new_tokens=256` (Limits response length without counting input tokens)
  * `min_new_tokens=20` (Forces a minimum output length)
* **Sampling & Creativity:**
  * `do_sample=True` (Enables probabilistic sampling instead of deterministic greedy search)
  * `temperature=0.7` (Higher values = more creative; lower values = more deterministic)
  * `top_p=0.9` (Nucleus sampling: considers tokens within cumulative probability mass)
  * `top_k=50` (Limits selection to top K highest probability tokens)
* **Repetition Penalties:**
  * `repetition_penalty=1.2` (Penalizes repeated words/phrases)
  * `no_repeat_ngram_size=3` (Prevents repeating exact phrase sequences of length N)
* **Streaming Responses:**
  * `streamer=TextStreamer(pipe.tokenizer)` (Prints tokens progressively to output console)

# Pending modification and testing

In [ ]:
from langchain import HuggingFacePipeline
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(model)

llm_gemma = HuggingFacePipeline(
    pipeline = pipeline,
    model_kwargs = {
        'temperature': 0,
        'max_length': 200,
        'do_sample': True,
        'top_k': 10,
        'num_return_sequences':1,
        'eos_token_id': tokenizer.eos_token_id
    }
)



In [ ]:
llm_gemma

In [ ]:
llm_gemma("What is AI")

In [ ]:
# TODO: Seguir en sección 1.3 de notebook_1

## 1.1 Uso de modelos Open Source de Hugging Face

Los modelos de Hugging Face requieren instalación de `einops`. Para utilizar `low_cpu_mem_usage=True` o `device_map` es necesario contar con `Accelerate` instalado: `pip install accelerate`.

In [ ]:
%%capture
!pip install -q transformers einops accelerate

Se debe tener en cuenta que entre más grande el `max_length`, es decir la cantidad de texto que podemos incluir en una consulta a nuestro modelo, más recursos computacionales se requieren. Vamos a usar los pipelines de Hugging Face: https://huggingface.co/docs/transformers/v4.30.0/main_classes/pipelines

In [ ]:
from transformers import AutoTokenizer, pipeline
import torch

# model = "tiiuae/falcon-40b-instruct"
# model = "stabilityai/stablelm-tuned-alpha-3b"
model = "tiiuae/falcon-7b-instruct"

tokenizer = AutoTokenizer.from_pretrained(model)

pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto"
)

A new version of the following files was downloaded from https://huggingface.co/tiiuae/falcon-7b-instruct:
- configuration_RW.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


A new version of the following files was downloaded from https://huggingface.co/tiiuae/falcon-7b-instruct:
- modelling_RW.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Xformers is not installed correctly. If you want to use memory_efficient_attention to accelerate training use the following command to install Xformers
pip install xformers.
The model 'RWForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'LlamaForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertForCausalLM', 'MvpForCausalLM', 'OpenLlamaForCausalLM', 'OpenAIGPTLMHeadModel', 'OPTForCausalLM', 'PegasusFor

Podemos enviar preguntas directamente al pipeline de Hugging Face para generar texto con nuestro modelo. Sin embargo, LangChain nos facilita la vida.

In [ ]:
type(pipeline)

transformers.pipelines.text_generation.TextGenerationPipeline

In [ ]:
from langchain import HuggingFacePipeline

llm_falcon = HuggingFacePipeline(
    pipeline = pipeline,
    model_kwargs = {
        'temperature': 0,
        'max_length': 200,
        'do_sample': True,
        'top_k': 10,
        'num_return_sequences':1,
        'eos_token_id': tokenizer.eos_token_id
    }
)



In [ ]:
llm_falcon

HuggingFacePipeline(cache=None, verbose=False, callbacks=None, callback_manager=None, tags=None, pipeline=<transformers.pipelines.text_generation.TextGenerationPipeline object at 0x7f85f83d6350>, model_id='gpt2', model_kwargs={'temperature': 0, 'max_length': 200, 'do_sample': True, 'top_k': 10, 'num_return_sequences': 1, 'eos_token_id': 11}, pipeline_kwargs=None)

In [ ]:
llm_falcon("What is AI?")

/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1259: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use a generation configuration file (see https://huggingface.co/docs/transformers/main_classes/text_generation)
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1353: UserWarning: Using `max_length`'s default (20) to control the generation length. This behaviour is deprecated and will be removed from the config in v5 of Transformers -- we recommend using `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'\nAI stands for Artificial Intelligence. It is a branch of computer science that focuses'

Los modelos de código abierto de Hugging Face son increíblemente poderosos. Sin embargo, al utilizarlos de esta manera, los descargamos y ejecutamos en nuestra propia máquina. Ahí es donde existen algunas complicaciones, ya que esto puede ser lento a menos que se cuente con el hardware adecuado.

Ahora piensa en modelos que provienen de API y servicios de OpenAI, Cohere y otros proveedores de modelos remotos (que normalmente no son de código abierto). La magia de estos modelos es que funcionan en sus servidores, no en nuestra máquina.

Es como si estuvieras invitado a una fiesta. Podrías hacer la fiesta en tu casa (como usar los modelos de Hugging Face en tu máquina), pero tendrías que hacer la limpieza antes y después, y preocuparte por la música, la comida, etc. En cambio, si la fiesta se celebra en un restaurante o salón dedicado a fiestas (como usar modelos de OpenAI o Cohere en sus servidores), solo tienes que llegar y disfrutar.

Por esto, vamos a seguir utilizando los modelos de la [API de OpenAI](https://platzi.com/cursos/openai). Todo lo que vamos a hacer a partir de ahora también se puede aplicar a los modelos descargados de Hugging Face.

### Hugging Face embeddings

In [ ]:
%%capture
!pip install sentence_transformers

In [ ]:
from langchain.embeddings import SentenceTransformerEmbeddings

embeddings_st = SentenceTransformerEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Otro modelo en español que podríamos usar es "symanto/sn-xlm-roberta-base-snli-mnli-anli-xnli"

In [ ]:
incrustaciones = embeddings_st.embed_documents(documentos_a_incrustar)
len(incrustaciones)

5

In [ ]:
len(incrustaciones[0])

384

In [ ]:
incrustacion = embeddings_st.embed_query(documentos_a_incrustar[0])


In [ ]:
len(incrustacion)

384

In [ ]:
%%capture
!pip install InstructorEmbedding sentence_transformers

In [ ]:
from langchain.embeddings import HuggingFaceInstructEmbeddings

# A junio de 2023 no hay modelos Instruct para español
embedding_instruct = HuggingFaceInstructEmbeddings(
    model_name="hkunlp/instructor-large",
    model_kwargs={"device":"cuda"}
)

# El device podría ser cpu

load INSTRUCTOR_Transformer
max_seq_length  512


In [ ]:
incrustaciones = embedding_instruct.embed_documents(documentos_a_incrustar)

In [ ]:
len(incrustaciones[4])

768

In [ ]:
incrustacion = embedding_instruct.embed_query(documentos_a_incrustar[0])

In [ ]:
len(incrustacion)

768

In [ ]:
embedding_instruct.client, embeddings_st.client

(INSTRUCTOR(
   (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: T5EncoderModel 
   (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False})
   (2): Dense({'in_features': 1024, 'out_features': 768, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity'})
   (3): Normalize()
 ),
 SentenceTransformer(
   (0): Transformer({'max_seq_length': 128, 'do_lower_case': False}) with Transformer model: BertModel 
   (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False})
 ))